# Track 2: Atlas-Free Coordinate GNN — Google Colab Training

**Goal**: Train `CoordGNN` on raw MNI peak coordinates (no atlas consulted).  
**Hardware**: A100 40GB (Runtime → Change runtime type → A100)  
**Output**: `best_coord_gnn.pt` and `last_coord_gnn.pt` saved to Google Drive

## Steps
1. Install dependencies & clone repo
2. Mount Google Drive for checkpoints
3. Load data (coordinates + SPECTER embeddings)
4. Build and cache KNN graphs
5. Phase 2–4: dataset → model → training
6. Phase 5–6: evaluation + attention analysis

## 0 · Runtime check

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No CUDA — switch to an A100 runtime via Runtime → Change runtime type")

## 1 · Install dependencies

In [ ]:
# Install PyTorch Geometric (match the torch version)
import subprocess, sys

torch_ver = torch.__version__.split('+')[0]   # e.g. '2.3.0'
cuda_tag  = 'cu121'                           # A100 default on Colab

pyg_wheels = f"https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html"

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch_geometric',
    'torch_scatter', 'torch_sparse', 'torch_cluster', 'torch_spline_conv',
    '-f', pyg_wheels
], check=True)

# Core neurovlm dependencies
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'nibabel', 'nilearn', 'nimare',
    'huggingface-hub', 'safetensors',
    'transformers', 'adapters',
    'pandas', 'pyarrow',
    'tqdm', 'umap-learn',
    'matplotlib', 'seaborn',
], check=True)

print('\nAll dependencies installed.')

In [ ]:
# Clone / pull the neurovlm repo
import os
from pathlib import Path

REPO_URL  = 'https://github.com/YOUR_USERNAME/neurovlm.git'  # ← set your fork URL
REPO_DIR  = Path('/content/neurovlm')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'pull'], cwd=str(REPO_DIR), check=True)

sys.path.insert(0, str(REPO_DIR / 'src'))
print(f'Repo ready at {REPO_DIR}')

## 2 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR       = Path('/content/drive/MyDrive/neurovlm_track2')
CHECKPOINT_DIR  = DRIVE_DIR / 'checkpoints/coord_gnn'
CACHE_DIR       = DRIVE_DIR / 'data/coord_graphs'

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Checkpoints → {CHECKPOINT_DIR}')
print(f'Graph cache → {CACHE_DIR}')

## 3 · Hyperparameters

In [ ]:
# ── A100 hyperparameters ──────────────────────────────────────────────────────
CFG = dict(
    # Graph construction
    K            = 7,      # KNN neighbors per node
    MAX_DIST_MM  = 30.0,   # prune edges beyond this mm distance
    # Model
    HIDDEN       = 128,    # hidden dim per head
    HEADS        = 8,      # attention heads
    DROPOUT      = 0.1,
    # Training
    N_EPOCHS     = 200,
    BATCH_SIZE   = 128,    # A100 can handle larger batches than MPS
    LR_GNN       = 1e-4,
    LR_PROJ      = 1e-5,
    WARMUP_EPOCHS= 15,
    TEMPERATURE  = 0.07,
    VAL_INTERVAL = 5,
    SEED         = 42,
)
print('Config:', CFG)

## 4 · Phase 1 — Load coordinates & statistics

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from neurovlm.retrieval_resources import _load_pubmed_coordinates
from neurovlm.gnn.coord_graph import normalize_coords

print('Loading peak coordinates …')
coords_df = _load_pubmed_coordinates()
print(f'  Shape: {coords_df.shape}  |  columns: {list(coords_df.columns)}')

# ── First 3 papers ────────────────────────────────────────────────────────
print('\nFirst 3 papers:')
for pmid, grp in list(coords_df.groupby('pmid'))[:3]:
    print(f'  PMID {pmid}: {len(grp)} peaks')
    print(f'    x∈[{grp.x.min():.1f}, {grp.x.max():.1f}]  '
          f'y∈[{grp.y.min():.1f}, {grp.y.max():.1f}]  '
          f'z∈[{grp.z.min():.1f}, {grp.z.max():.1f}]')

In [ ]:
# ── Peak count statistics ─────────────────────────────────────────────────
peak_counts = coords_df.groupby('pmid').size()
print(f'Peak counts across {len(peak_counts):,} papers')
print(f'  min={peak_counts.min()}  max={peak_counts.max()}')
print(f'  mean={peak_counts.mean():.1f}  median={peak_counts.median():.0f}')
print(f'  p5={peak_counts.quantile(0.05):.0f}  p95={peak_counts.quantile(0.95):.0f}')
print(f'  Papers with <3 peaks: {(peak_counts<3).sum()} '
      f'({100*(peak_counts<3).mean():.1f}%)')

fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(peak_counts.clip(upper=100), bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
ax.set_xlabel('Peaks per paper')
ax.set_ylabel('# papers')
ax.set_title('Distribution of peak counts (clipped at 100)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Normalization check ───────────────────────────────────────────────────
coords_arr = coords_df[['x', 'y', 'z']].values
norm_arr   = normalize_coords(coords_arr)
print(f'Normalized range: [{norm_arr.min():.4f}, {norm_arr.max():.4f}]')
outliers = (np.abs(norm_arr) > 1.05).any(axis=1).sum()
print(f'Outliers beyond ±1.05 (clipped): {outliers}')

# ── Duplicate check ───────────────────────────────────────────────────────
dup_papers, max_dups = 0, 0
for _, grp in coords_df.groupby('pmid'):
    raw = grp[['x','y','z']].values
    n_dups = len(raw) - len(np.unique(raw, axis=0))
    if n_dups > 0:
        dup_papers += 1
        max_dups = max(max_dups, n_dups)
print(f'Papers with duplicate peaks: {dup_papers} (max dups: {max_dups})')

In [ ]:
# ── k sensitivity analysis: avg degree and filter rate at k=5,7,10 ────────
import random
from neurovlm.gnn.coord_graph import coords_to_graph

sample_pmids = random.sample(list(coords_df['pmid'].unique()), 100)
sample_df = coords_df[coords_df['pmid'].isin(sample_pmids)]

print('k | avg_degree | pct_filtered_by_dist')
print('--+------------+----------------------')
for k_val in [5, 7, 10]:
    degrees, filtered_fracs = [], []
    for pmid, grp in sample_df.groupby('pmid'):
        raw = np.unique(grp[['x','y','z']].values, axis=0)
        norm = normalize_coords(raw)
        g = coords_to_graph(norm, k=k_val, max_dist_mm=30.0)
        n_nodes = g.x.shape[0]
        n_edges = g.edge_index.shape[1]
        # Max possible edges with KNN = n_nodes * min(k_val, n_nodes-1)
        max_edges = n_nodes * min(k_val, n_nodes - 1) if n_nodes > 1 else 0
        degrees.append(n_edges / max(n_nodes, 1))
        if max_edges > 0:
            filtered_fracs.append(1.0 - n_edges / max_edges)
    print(f' {k_val} | {np.mean(degrees):10.2f} | {np.mean(filtered_fracs)*100:20.1f}%')

print('\nTarget: avg_degree in [4,12], pct_filtered < 20%')

## 5 · Phase 2 — Dataset

In [ ]:
from neurovlm.data import load_dataset, load_latent

print('Loading SPECTER text embeddings …')
text_latents = load_latent('pubmed_text')
pubmed_df    = load_dataset('pubmed_text')

# Resolve to (tensor, pmid_array) form
if isinstance(text_latents, tuple):
    text_tensor, pmids_arr = text_latents
    unique_pmids = np.asarray(pmids_arr).astype(str)
elif isinstance(text_latents, dict):
    pmid_list    = list(text_latents.keys())
    text_tensor  = torch.stack([torch.tensor(text_latents[p], dtype=torch.float32)
                                for p in pmid_list])
    unique_pmids = np.array([str(p) for p in pmid_list])
else:
    text_tensor  = text_latents if isinstance(text_latents, torch.Tensor) \
                   else torch.tensor(text_latents)
    unique_pmids = pubmed_df['pmid'].astype(str).values[:len(text_tensor)] \
                   if 'pmid' in pubmed_df.columns \
                   else np.arange(len(text_tensor)).astype(str)

print(f'  text_tensor: {text_tensor.shape}')
print(f'  unique_pmids: {len(unique_pmids):,}')

In [ ]:
from neurovlm.gnn.coord_dataset import CoordGraphDataset

print('Building CoordGraphDataset …')
print('  Graphs will be cached to Google Drive on first run.')
print(f'  Cache dir: {CACHE_DIR}')

ds = CoordGraphDataset(
    coords_df    = coords_df,
    text_embeddings = text_tensor,
    unique_pmids = unique_pmids,
    cache_dir    = str(CACHE_DIR),
    k            = CFG['K'],
    max_dist_mm  = CFG['MAX_DIST_MM'],
)
print(f'Dataset size: {len(ds):,} papers')

In [ ]:
# Verify PyG batching works
from torch_geometric.loader import DataLoader
probe = next(iter(DataLoader(ds, batch_size=4, shuffle=False)))
print('PyG batch check (batch_size=4):')
print(f'  batch.x.shape           = {probe.x.shape}   (total_nodes × 5)')
print(f'  batch.edge_index.shape  = {probe.edge_index.shape}')
print(f'  batch.edge_attr.shape   = {probe.edge_attr.shape}  (E × 4)')
print(f'  batch.y.shape           = {probe.y.shape}')
print(f'  batch.batch.unique()    = {probe.batch.unique().tolist()}')
assert probe.x.shape[1] == 5, 'Node feature dim must be 5'
assert probe.edge_attr.shape[1] == 4, 'Edge feature dim must be 4'
print('✓ All shapes correct')

In [ ]:
torch.manual_seed(CFG['SEED'])
train_ds, val_ds, test_ds = ds.split(val_frac=0.1, test_frac=0.1, seed=CFG['SEED'])
print(f'Split: train={len(train_ds):,}  val={len(val_ds):,}  test={len(test_ds):,}')

## 6 · Phase 3 — Model

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Data, Batch

from neurovlm.gnn.coord_model import CoordGNN
from neurovlm.gnn.model import TextProjHead

brain_encoder = CoordGNN(
    in_dim  = 5,
    hidden  = CFG['HIDDEN'],
    heads   = CFG['HEADS'],
    out_dim = 384,
    dropout = CFG['DROPOUT'],
)
text_proj = TextProjHead(in_dim=768, hidden_dim=512, out_dim=384)

n_brain = brain_encoder.count_parameters()
n_text  = sum(p.numel() for p in text_proj.parameters())
print(f'CoordGNN params     : {n_brain:,}')
print(f'TextProjHead params : {n_text:,}')

assert 500_000 <= n_brain <= 5_000_000, \
    f'Model size {n_brain:,} out of expected range [500K, 5M]'
print('✓ Parameter count in range [500K, 5M]')

# Shape check
dummy = Data(
    x=torch.randn(10, 5),
    edge_index=torch.tensor([[0,1,2],[1,2,0]], dtype=torch.long),
    edge_attr=torch.randn(3, 4),
)
b = Batch.from_data_list([dummy, dummy])
out = brain_encoder(b.x, b.edge_index, b.edge_attr, b.batch)
assert out.shape == (2, 384)
print(f'✓ Forward pass: input (20, 5) → output {tuple(out.shape)}')

## 7 · Phase 4 — Training

In [ ]:
from neurovlm.gnn.coord_train import CoordTrainer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on: {DEVICE}')

trainer = CoordTrainer(
    brain_encoder  = brain_encoder,
    text_proj      = text_proj,
    lr_gnn         = CFG['LR_GNN'],
    lr_proj        = CFG['LR_PROJ'],
    batch_size     = CFG['BATCH_SIZE'],
    n_epochs       = CFG['N_EPOCHS'],
    warmup_epochs  = CFG['WARMUP_EPOCHS'],
    temperature    = CFG['TEMPERATURE'],
    device         = DEVICE,
    val_interval   = CFG['VAL_INTERVAL'],
    checkpoint_dir = str(CHECKPOINT_DIR),
    verbose        = True,
)

trainer.fit(train_ds, val_ds)

In [ ]:
# ── Training curve ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(trainer.history['train_loss'])
axes[0].set_title('Train Loss (InfoNCE)')
axes[0].set_xlabel('Epoch')

axes[1].plot(trainer.history['val_recall_t2i'], label='t2i')
axes[1].plot(trainer.history['val_recall_i2t'], label='i2t')
axes[1].set_title('Val Recall AUC')
axes[1].set_xlabel(f'Validation step (every {CFG["VAL_INTERVAL"]} epochs)')
axes[1].legend()

axes[2].plot(trainer.history['embed_sim'], color='red')
axes[2].axhline(0.95, color='black', linestyle='--', label='collapse threshold')
axes[2].set_title('Embedding Similarity (collapse monitor)')
axes[2].set_xlabel(f'Validation step')
axes[2].legend()

plt.tight_layout()
plt.savefig(str(DRIVE_DIR / 'training_curve.png'), dpi=150)
plt.show()

## 8 · Phase 5 — Evaluation

In [ ]:
from neurovlm.metrics import recall_at_k, recall_curve

trainer.restore_best()

results = {}
for split_name, ds in [('Val', val_ds), ('Test', test_ds)]:
    brain_emb, text_emb = trainer.collect_embeddings(ds)
    brain_n = F.normalize(brain_emb, dim=1)
    text_n  = F.normalize(text_emb,  dim=1)
    sim     = text_n @ brain_n.T

    r1    = recall_at_k(sim, 1)
    r5    = recall_at_k(sim, 5)
    r10   = recall_at_k(sim, 10)
    t2i, i2t = recall_curve(text_emb, brain_emb)
    auc   = float((t2i.mean() + i2t.mean()) / 2)

    results[split_name] = dict(r1=r1, r5=r5, r10=r10, auc=auc)
    print(f'\n{split_name} ({len(ds)} papers):')
    print(f'  recall@1  = {r1:.4f}')
    print(f'  recall@5  = {r5:.4f}')
    print(f'  recall@10 = {r10:.4f}')
    print(f'  AUC       = {auc:.4f}  (NeuroVLM MLP baseline ≈ 0.81)')

print('\n┌───────────────────────────┬──────────┬────────────┐')
print('│ Model                     │ Test AUC │ Atlas-free │')
print('├───────────────────────────┼──────────┼────────────┤')
print('│ NeuroVLM MLP (baseline)   │ 0.81     │ No         │')
print('│ Track 1 DiFuMo GAT        │ (run it) │ No         │')
print(f'│ Track 2 Coord GNN (ours)  │ {results["Test"]["auc"]:.4f}   │ Yes        │')
print('└───────────────────────────┴──────────┴────────────┘')

## 9 · Phase 6 — Attention Analysis

In [ ]:
print('Extracting attention weights for 10 test papers …')
snapshots = trainer.get_attention_snapshot(test_ds, n_samples=10)

for snap in snapshots:
    n_peaks = snap['node_coords_mni'].shape[0]
    print(f'\nPaper {snap["paper_idx"]} ({n_peaks} peaks):')
    for e in snap['top_edges']:
        s = [f'{v:.1f}' for v in e['src_mni']]
        d = [f'{v:.1f}' for v in e['dst_mni']]
        dist_mm = float(np.linalg.norm(
            np.array(e['src_mni']) - np.array(e['dst_mni'])
        ))
        print(f'  [{', '.join(s)}] → [{', '.join(d)}]  '
              f'attn={e["weight"]:.4f}  dist={dist_mm:.1f}mm')

## 10 · Failure Analysis

In [ ]:
brain_emb, text_emb = trainer.collect_embeddings(test_ds)
brain_n = F.normalize(brain_emb, dim=1)
text_n  = F.normalize(text_emb,  dim=1)
sim     = text_n @ brain_n.T

top10_hits = (sim.argsort(dim=1, descending=True)[:, :10] ==
              torch.arange(len(sim)).unsqueeze(1)).any(dim=1)

worst_pos = (~top10_hits).nonzero(as_tuple=True)[0][:20].tolist()
print(f'Papers with recall@10=0: {(~top10_hits).sum()} / {len(sim)}')
print(f'Worst 20 positions: {worst_pos}')

# Peak count for worst papers
from torch_geometric.loader import DataLoader as PyGDataLoader
worst_loader = PyGDataLoader(
    [test_ds[i] for i in worst_pos[:20]], batch_size=1, shuffle=False
)
peak_counts_worst = []
for g in worst_loader:
    peak_counts_worst.append(g.x.shape[0])

print(f'\nWorst-paper peak counts:')
print(f'  mean={np.mean(peak_counts_worst):.1f}  '
      f'min={min(peak_counts_worst)}  max={max(peak_counts_worst)}')
print('(Papers with very few peaks tend to have high failure rates.)')

## 11 · UMAP Visualization

In [ ]:
import umap
import matplotlib.pyplot as plt
import seaborn as sns

brain_emb, text_emb = trainer.collect_embeddings(test_ds)
brain_n = F.normalize(brain_emb, dim=1).cpu().numpy()

# Sample 500 points for speed
N_UMAP = min(500, len(brain_n))
rng = np.random.default_rng(42)
idx = rng.choice(len(brain_n), N_UMAP, replace=False)
sample = brain_n[idx]

print(f'Running UMAP on {N_UMAP} embeddings …')
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
emb_2d  = reducer.fit_transform(sample)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(emb_2d[:, 0], emb_2d[:, 1], s=5, alpha=0.6, c='steelblue')
ax.set_title(f'UMAP of Track 2 CoordGNN brain embeddings ({N_UMAP} test papers)')
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')
plt.tight_layout()
plt.savefig(str(DRIVE_DIR / 'umap_coord_gnn.png'), dpi=150)
plt.show()
print(f'Saved to {DRIVE_DIR}/umap_coord_gnn.png')

## 12 · Save final artifacts to Drive

In [ ]:
import json

# Save training history
history_path = DRIVE_DIR / 'training_history.json'
with open(history_path, 'w') as f:
    json.dump({
        k: [float(v) for v in vals]
        for k, vals in trainer.history.items()
    }, f, indent=2)
print(f'History saved → {history_path}')

# Save results
results_path = DRIVE_DIR / 'eval_results.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Eval results saved → {results_path}')

# List all files in DRIVE_DIR
for p in sorted(DRIVE_DIR.rglob('*')):
    if p.is_file():
        size_mb = p.stat().st_size / 1e6
        print(f'  {p.relative_to(DRIVE_DIR)}  ({size_mb:.1f} MB)')